In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, mean_squared_error, root_mean_squared_error, confusion_matrix
from scipy.stats import pearsonr
from scipy.spatial.distance import pdist, squareform
!pip install tensorflow-gpu
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Input, Dropout, Layer
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import register_keras_serializable, custom_object_scope
from google.colab import drive
drive.mount('/content/drive')

  Using cached tensorflow-gpu-2.12.0.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# df_org = pd.read_csv('forestfires.csv')
# df = df_org.copy()
# month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
#              'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
# weeks_map = {'sun': 0, 'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6}

# df['month_num'] = df['month'].map(month_map)
# df['week_num'] = df['day'].map(weeks_map)


# length = len(df)
# df['index'] = df.index
# # df['label'] = 1  # 1 means normal, 0 mean outlier
# columns_to_remove = {'month', 'day', 'index','label'}

In [ ]:
class DataPreprocessor:
    def __init__(self, df, sel_feature=None, sigma=0.2, OP=1, categorical_maps=None, label_col=None):

        self.df = df.copy()
        self.df_org = df.copy()
        self.sel_feature = sel_feature
        self.sigma = sigma
        self.OUTLIER_PERCENTAGE = OP
        self.length = len(self.df)
        self.COUNT_OUTLIER = int(self.OUTLIER_PERCENTAGE / 100 * self.length)
        self.random_indices = np.random.choice(self.df.index, self.COUNT_OUTLIER, replace=False)
        self.features = []
        self.categorical_maps = categorical_maps or {}
        self.label_col = label_col


    def prepare_data(self):
        self.df = self.df_org.copy()

        # Handle categorical columns
        for col, mapping in self.categorical_maps.items():
            self.df[f'{col}_num'] = self.df[col].map(mapping)
            self.df = self.df.drop(col, errors='ignore')
        # Initialize labels if not present
        if self.label_col not in self.df.columns:
            self.df['label'] = 1  # 1 means normal, 0 means outlier

        # Convert selected feature to float
        if self.sel_feature:
            self.df[self.sel_feature] = self.df[self.sel_feature].astype(float)

            # Inject noise only if feature is specified
            for index in self.random_indices:
                df_1 = self.df.iloc[index].copy()
                temp = df_1[self.sel_feature]
                sigma_T = self.sigma * temp
                noise_T = random.gauss(0, sigma_T)
                temp += noise_T
                df_1[self.sel_feature] = temp
                df_1['label'] = 0
                self.df.loc[index] = df_1

        # Define features to use (exclude non-feature columns)
        non_feature_cols = {'label'}
        if self.label_col:
            non_feature_cols.add(self.label_col)
        self.features = [col for col in self.df.columns if col not in non_feature_cols]

        self.df = self.df.sample(frac=1).reset_index(drop=True)

        X = self.df[self.features]
        y = self.df['label']
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.20, random_state=369, shuffle=False)
        return X_scaled, X_train, X_test, y_train, y_test, self.features
        # return train_test_split(X_scaled, y, test_size=0.20, random_state=369, shuffle=False)

# Enhanced DataPreprocessor for all datasets
class EnhancedDataPreprocessor(DataPreprocessor):
    @staticmethod
    def get_dataset_config(dataset_name, df=None):
        """Returns only the config for the requested dataset"""
        configs = {
            'forestfires': {
                'categorical_maps': {
                    'month': {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
                              'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12},
                    'day': {'sun':0, 'mon':1, 'tue':2, 'wed':3, 'thu':4, 'fri':5, 'sat':6}
                },
                'label_col': None
            },
            'pems': {
                'categorical_maps': {},
                'label_col': None
            }
        }

        # Special handling for UNSW as it needs the DataFrame
        if dataset_name == 'unsw' and df is not None:
            configs['unsw'] = {
                'categorical_maps': {
                    'proto': {proto:i for i, proto in enumerate(df['proto'].unique())},
                    'service': {service:i for i, service in enumerate(df['service'].unique())},
                    'attack_cat': {cat:i for i, cat in enumerate(df['attack_cat'].unique())}
                },
                'label_col': 'label'
            }
            return configs.get(dataset_name, {})

        return configs.get(dataset_name, {})

    def __init__(self, df, dataset_name, sel_feature=None, sigma=0.2, OP=1):
        config = self.get_dataset_config(dataset_name, df)
        super().__init__(
            df=df,
            sel_feature=sel_feature,
            sigma=sigma,
            OP=OP,
            categorical_maps=config.get('categorical_maps', None),
            label_col=config.get('label_col', None)
        )

In [ ]:
@register_keras_serializable()
class AutoEncoder(Model):
    def __init__(self, output_units, neck=8,  **kwargs):
        super().__init__( **kwargs)
        self.output_units = output_units
        self.neck = neck
        self.threshold = None
        self.encoder = tf.keras.Sequential([
            Dense(64, activation='relu'),
            Dropout(0.1),
            Dense(32, activation='relu'),
            Dropout(0.1),
            Dense(16, activation='relu'),
            Dropout(0.1),
            Dense(neck, activation='relu')
        ])
        self.decoder = tf.keras.Sequential([
            Dense(16, activation='relu'),
            Dropout(0.1),
            Dense(32, activation='relu'),
            Dropout(0.1),
            Dense(64, activation='relu'),
            Dropout(0.1),
            Dense(output_units, activation='sigmoid')
        ])

    def predict_label(self, inputs):
        encoded = self.encoder(inputs)
        raw = self.decoder(encoded)
        test_errors = tf.keras.losses.msle(raw, inputs)
        ae_preds = (pd.Series(test_errors) > self.threshold).map(lambda x: 0.0 if x else 1.0).astype('float32')
        return ae_preds

    def call(self, inputs):
        encoded = self.encoder(inputs)
        return self.decoder(encoded)

    def get_config(self):
        config = super().get_config()
        config.update({
            'output_units': self.output_units,
            'neck': self.neck
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

class ModelTrainer:
    def __init__(self):
        self.models = {}
        self.ae_threshold = None


    def train_or_load(self, X_train, y_train, dataset_name, feature, model_type, sigma, OP):
        """Smart training that checks cache first"""
        # Try loading cached model
        cached_model = load_model(dataset_name, feature, model_type, sigma, OP)
        if cached_model is not None:
            self.models[f"{model_type}_{feature}"] = cached_model
            if model_type == 'ae':
                self.ae_threshold = cached_model.threshold
            return

        # Train new model if not cached
        print(f"Training new {model_type} model for {feature}...")
        if model_type == 'rf':
            self.train_rf(X_train, y_train, feature)
        elif model_type == 'knn':
            self.train_knn(X_train, y_train, feature)
        elif model_type == 'ae':
            self.train_autoencoder(X_train, y_train, feature, OP)

        # Save the newly trained model
        model = self.models[f"{model_type}_{feature}"]
        save_path = save_model(model, dataset_name, feature, model_type, sigma, OP)
        print(f"Saved new model to: {save_path}")


    def train_rf(self, X_train, y_train, feature_name):
        # X_train_feature = X_train[:, feature_index].reshape(-1, 1)
        rf = RandomForestClassifier(n_estimators=10, random_state=42)
        rf.fit(X_train, y_train)
        self.models[f"rf_{feature_name}"] = rf

    def train_knn(self, X_train, y_train, feature_name):
        # X_train_feature = X_train[:, feature_index].reshape(-1, 1)
        knn = KNeighborsClassifier(n_neighbors=5)
        knn.fit(X_train, y_train)
        self.models[f"knn_{feature_name}"] = knn

    def train_autoencoder(self, X_train, y_train, feature_name, OP):
        model = AutoEncoder(output_units=X_train.shape[1])

        model.compile(optimizer='adam', loss='msle', metrics=['mse'])
        early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')

        # X_train_feature = X_train[:, feature_index].reshape(-1, 1)  # Shape (batch_size, 1)
        # X_test_feature = X_test[:, feature_index].reshape(-1, 1)

        # history = model.fit(X_train, X_train, epochs=100, batch_size=128, validation_split=0.1, verbose=0, callbacks=[early_stop])
        history = model.fit(X_train, X_train, epochs=100, batch_size=128, validation_split=0.1, verbose=0, callbacks=[early_stop])

        reconstructions = model.predict(X_train)
        reconstruction_errors = tf.keras.losses.msle(reconstructions, X_train)
        self.ae_threshold = np.percentile(reconstruction_errors, 100 - OP)

        model.threshold = self.ae_threshold
        self.models[f"ae_{feature_name}"] = model





In [ ]:
import pickle
import os
import hashlib
from datetime import datetime

MODEL_CACHE_DIR = "/content/drive/My Drive/MTP_DATA/model_cache/"
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)


def clear_gpu_memory():
    tf.keras.backend.clear_session()
    cuda.select_device(0)
    cuda.close()
    gc.collect()

def get_model_hash(dataset_name, feature, model_type, sigma, OP):
    """Generate unique hash for model identification"""
    params = f"{dataset_name}_{feature}_{model_type}_{sigma}_{OP}"
    return hashlib.md5(params.encode()).hexdigest(), params

def save_model(model, dataset_name, feature, model_type, sigma, OP):
    model_hash = get_model_hash(dataset_name, feature, model_type, sigma, OP)
    metadata = {
        'dataset': dataset_name,
        'feature': feature,
        'model_type': model_type,
        'sigma': sigma,
        'outlier_percentage': OP,
        'timestamp': datetime.now().isoformat(),
        'model_hash': model_hash
    }

    save_path = os.path.join(MODEL_CACHE_DIR, f"{model_hash}.pkl")

    if isinstance(model, (Model, Layer)):
        # Save Keras model
        model_path = os.path.join(MODEL_CACHE_DIR, f"{model_hash}.keras")
        model.save(model_path, save_format='tf')

        # Save additional attributes
        model_attrs = {
            'threshold': model.threshold
        }

        with open(save_path, 'wb') as f:
            pickle.dump({
                'metadata': metadata,
                'model_attrs': model_attrs,
                'keras_path': model_path
            }, f)
    else:
        with open(save_path, 'wb') as f:
            pickle.dump({
                'model': model,
                'metadata': metadata
            }, f)

    return save_path

def load_model(dataset_name, feature, model_type, sigma, OP):
    model_hash = get_model_hash(dataset_name, feature, model_type, sigma, OP)
    save_path = os.path.join(MODEL_CACHE_DIR, f"{model_hash}.pkl")

    if not os.path.exists(save_path):
        return None

    with open(save_path, 'rb') as f:
        data = pickle.load(f)

    print(f"Loading cached {model_type} model: {data['metadata']}")

    if 'keras_path' in data:
        try:
            model = tf.keras.models.load_model(
                data['keras_path'],
                custom_objects={'AutoEncoder': AutoEncoder}
            )
            # Restore custom attributes
            for attr, value in data['model_attrs'].items():
                setattr(model, attr, value)
            return model
        except Exception as e:
            print(f"Error loading Keras model: {e}")
            return None
    else:
        return data['model']

def inspect_model_cache():
    """View all cached models and their metadata"""
    cached_models = []
    for fname in os.listdir(MODEL_CACHE_DIR):
        if fname.endswith('.pkl'):
            with open(os.path.join(MODEL_CACHE_DIR, fname), 'rb') as f:
                try:
                    data = pickle.load(f)
                    cached_models.append(data['metadata'])
                except:
                    continue
    return pd.DataFrame(cached_models)



In [ ]:
class Evaluator:
    @staticmethod
    def evaluate(models, X_test, feature_index, y_test):
        print("Evaluation Metrics:")
        # comp_indices = {}
        results = {}

        for name, model in models.items():
            # X_test_feature = X_test[:, feature_index].reshape(-1, 1)
            if name.startswith("ae"):
                preds = model.predict_label(X_test)
            else:
              preds = model.predict(X_test)
        #     comp_indices[name] = [i for i, j in enumerate(preds) if j == 0]
        #     print(f"{name} Accuracy: {accuracy_score(y_test, preds):.4f}")
        #     print(f"{name} F1 Score: {f1_score(y_test, preds):.4f}")
        #     print(f"{name} Precision: {precision_score(y_test, preds):.4f}")
        #     print(f"{name} Recall: {recall_score(y_test, preds):.4f}")
        #     print("__" * 50)
        # return comp_indices
              results[name] = {
                'accuracy': accuracy_score(y_test, preds),
                'f1': f1_score(y_test, preds),
                'precision': precision_score(y_test, preds),
                'recall': recall_score(y_test, preds)
              }
        return results

In [ ]:
tf.random.set_seed(36)
class FIDRecovery:
    def __init__(self):
        self.device = self._detect_hardware()
        self.strategy = self._initialize_strategy()  # Add this method
        self.small_dataset_threshold = 5000
        # Pre-compile TF functions to avoid retracing
        self.compute_fid_tf = tf.function(self.compute_fid,
                # input_signature=[ tf.TensorSpec(shape=[None], dtype=tf.float32),  # org_tensor
                # tf.TensorSpec(shape=[None], dtype=tf.int32)     # comp_indices
             reduce_retracing=True )

    def _initialize_strategy(self):
        if self.device == 'tpu':
            resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
            tf.config.experimental_connect_to_cluster(resolver)
            tf.tpu.experimental.initialize_tpu_system(resolver)
            return tf.distribute.TPUStrategy(resolver)
        elif self.device == 'gpu':
            return tf.distribute.MirroredStrategy()
        else:
            return tf.distribute.get_strategy()  # Adjust based on your needs

    def _detect_hardware(self):
        """Detect available hardware and set appropriate strategies"""
        try:
            if tf.config.list_physical_devices('TPU'):
                # resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
                # tf.config.experimental_connect_to_cluster(resolver)
                # tf.tpu.experimental.initialize_tpu_system(resolver)
                return 'tpu'
            elif tf.config.list_physical_devices('GPU'):
                return 'gpu'
        except:
            pass
        return 'cpu'

    def _should_use_gpu(self, data_size):
        """Heuristic to determine if GPU should be used"""
        if self.device == 'cpu':
            return False
        return data_size > self.small_dataset_threshold

    def calculate_correlation_weights(self, X, target_feature_index, method='pearson'):
        """Automatically selects optimal implementation"""
        if not self._should_use_gpu(X.shape[0]):
            # redictiing to CPU fallback for small datasets
            df_local = pd.DataFrame(X)
            correlation_series = df_local.corr(method=method)[target_feature_index]
            feature_weights = correlation_series.abs().values
            feature_weights[target_feature_index] = 1.0
            return feature_weights / feature_weights.sum()

        # redirect to GPU implementation for pearson
        if method == 'pearson':
            return self._gpu_pearson_correlation(X, target_feature_index)
        else:
            # fallback to CPU for other methods
            return self._cpu_correlation(X, target_feature_index, method)

    def _gpu_pearson_correlation(self, X, target_feature_index):
        """GPU-optimized Pearson correlation"""
        X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
        target = X_tensor[:, target_feature_index]

        # Center the data
        centered = X_tensor - tf.reduce_mean(X_tensor, axis=0)
        target_centered = target - tf.reduce_mean(target)

        # Compute covariance and variances
        cov = tf.reduce_mean(centered * target_centered[:, tf.newaxis], axis=0)
        var_target = tf.reduce_mean(tf.square(target_centered))
        var_features = tf.reduce_mean(tf.square(centered), axis=0)

        # Compute correlations
        corr = cov / tf.sqrt(var_features * var_target)
        corr = tf.where(tf.math.is_finite(corr), corr, tf.zeros_like(corr))

        # Ensure self-correlation is 1.0
        corr = tf.tensor_scatter_nd_update(corr, [[target_feature_index]], [1.0])
        return (corr / tf.reduce_sum(tf.abs(corr))).numpy()

    def _cpu_correlation(self, X, target_feature_index, method):
        """Fallback CPU implementation"""
        if method == 'distance':
            return self.calculate_distance_correlation_weights(X, target_feature_index)
        else:
            df_local = pd.DataFrame(X)
            correlation_series = df_local.corr(method=method)[target_feature_index]
            feature_weights = correlation_series.abs().values
            feature_weights[target_feature_index] = 1.0
            return feature_weights / feature_weights.sum()

    def calculate_distance_correlation_weights(self, X, target_feature_index):
        """Automatically selects CPU/GPU implementation"""
        if not self._should_use_gpu(X.shape[0]):
            # redict to CPU implementation for small datasets
            target_feature = X[:, target_feature_index]
            weights = []
            for i in range(X.shape[1]):
                if i == target_feature_index:
                    weights.append(1.0)
                else:
                    weights.append(self._cpu_distance_correlation(X[:, i], target_feature))
            weights = np.array(weights)
            return weights / np.sum(weights)
        else:
            # redirecting to GPU implementation
            return self._gpu_distance_correlation_weights(X, target_feature_index)

    def _cpu_distance_correlation(self, x, y):
        """Original CPU implementation"""
        def _dcov(x, y):
            a = squareform(pdist(x[:, None], 'euclidean'))
            b = squareform(pdist(y[:, None], 'euclidean'))
            A = a - a.mean(axis=0)[None, :] - a.mean(axis=1)[:, None] + a.mean()
            B = b - b.mean(axis=0)[None, :] - b.mean(axis=1)[:, None] + b.mean()
            return np.sqrt((A * B).mean())
        return _dcov(x, y) / np.sqrt(_dcov(x, x) * _dcov(y, y))

    def _gpu_distance_correlation_weights(self, X, target_feature_index):
        """Optimized GPU implementation"""
        with self.strategy.scope():
            X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
            n_features = X_tensor.shape[1]
            batch_size = 512  # Adjust based on your GPU memory

            # Function to compute distance matrices for a batch
            def compute_dist_mats(batch):
                expanded = tf.expand_dims(batch, 1)
                diff = expanded - tf.transpose(expanded, perm=[1, 0, 2])
                return tf.norm(diff, axis=2)

            # Process in batches
            dist_mats = tf.concat([compute_dist_mats(X_tensor[i:i + batch_size])
                for i in range(0, X_tensor.shape[0], batch_size)], axis=0)

            # Center all distance matrices
            means_i = tf.reduce_mean(dist_mats, axis=1, keepdims=True)
            means_j = tf.reduce_mean(dist_mats, axis=2, keepdims=True)
            grand_mean = tf.reduce_mean(dist_mats)
            centered_mats = dist_mats - means_i - means_j + grand_mean

            # Compute correlations
            target_centered = centered_mats[..., target_feature_index]
            covariances = tf.reduce_mean(centered_mats * target_centered[..., tf.newaxis], axis=[0,1])
            var_target = tf.reduce_mean(tf.square(target_centered))
            variances = tf.reduce_mean(tf.square(centered_mats), axis=[0,1])

            # Handle numerical stability
            denominators = tf.sqrt(var_target * variances)
            weights = tf.where(denominators > 1e-8,
                              covariances / denominators,
                              tf.zeros_like(covariances))

            # Ensure self-correlation is 1.0
            weights = tf.tensor_scatter_nd_update(weights,
                                                [[target_feature_index]],
                                                [1.0])
            return (weights / tf.reduce_sum(weights)).numpy()

    def calculate_shallow_nn_weights(self, X, target_feature_index):
        """Automatically selects CPU/GPU implementation"""
        if not self._should_use_gpu(X.shape[0]):
            target_feature = X[:, target_feature_index]
            other_features = np.delete(X, target_feature_index, axis=1)

            nn = MLPRegressor(hidden_layer_sizes=(10,), max_iter=1000,
                             random_state=42)
            nn.fit(other_features, target_feature)

            weights = np.abs(nn.coefs_[0]).mean(axis=1)
            weights = np.insert(weights, target_feature_index, 1.0)
            return weights / np.sum(weights)
        else:
            # GPU implementation
            return self._gpu_shallow_nn(X, target_feature_index)

    def _gpu_shallow_nn(self, X, target_feature_index):
        """GPU-optimized shallow NN"""
        with self.strategy.scope():
            # Force float32 on TPU because its float64 takes hell lot of time(for using gpu its kinda problematic tho)
            policy = tf.keras.mixed_precision.Policy('mixed_float32' if self.device == 'tpu' else 'mixed_float16')
            tf.keras.mixed_precision.set_global_policy(policy)
            X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
            target = X_tensor[:, target_feature_index]
            features = tf.concat([X_tensor[:, :target_feature_index],
                            X_tensor[:, target_feature_index+1:]], axis=1)

            model = tf.keras.Sequential([
                tf.keras.layers.Dense(10, activation='relu'),
                tf.keras.layers.Dense(1)])
            model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
            # TPUs require fixed batch sizes
            batch_size = 512 * self.strategy.num_replicas_in_sync
            early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5, restore_best_weights=True)
            model.fit(features, target, epochs=100, validation_split=0.3, batch_size=batch_size, verbose=0, callbacks=[early_stop])

            weights = tf.reduce_mean(tf.abs(model.layers[0].weights[0]), axis=1)
            weights = tf.concat([
                weights[:target_feature_index],
                [1.0],
                weights[target_feature_index:]
            ], axis=0)

            return (weights / tf.reduce_sum(weights)).numpy()

    def FID(self, org_data, compromised_indices, X=None, target_feature_index=None, method=None):
        """Main entry point with automatic implementation selection"""
        if not self._should_use_gpu(len(org_data)):
            return self._cpu_FID(org_data, compromised_indices, X, target_feature_index, method)
        else:
            return self._gpu_FID(org_data, compromised_indices, X, target_feature_index, method)

    def _cpu_FID(self, org_data, compromised_indices, X=None, target_feature_index=None, method=None):
        """Original CPU implementation with all methods"""
        if method is None:
            # Base FID
            compromised_indices_set = set(compromised_indices)
            observed_data = [val for ind, val in enumerate(org_data) if ind not in compromised_indices_set]

            mean = np.mean(observed_data)
            a, b = min(observed_data), max(observed_data)
            t = len(compromised_indices)
            l = (b - a) / t
            V = [(a + (s - 1) * l + a + s * l) / 2 for s in range(1, t + 1)]

            R = []
            for v in V:
                contribution_weights = [1 - abs(i - v) / l if abs(i - v) <= l else 0 for i in observed_data]
                sum_weights = sum(contribution_weights)
                if sum_weights == 0:
                    R.append(mean)
                else:
                    weighted_sum = sum(w * val for w, val in zip(contribution_weights, observed_data))
                    R.append(weighted_sum / sum_weights)

            recovered_data = np.copy(org_data)
            for i, idx in enumerate(compromised_indices):
                recovered_data[idx] = R[i]
            return recovered_data
        else:
            if method == 'pearson':
                feature_weights = self.calculate_correlation_weights(X, target_feature_index, 'pearson')
            elif method == 'distance':
                feature_weights = self.calculate_distance_correlation_weights(X, target_feature_index)
            elif method == 'shallow_nn':
                feature_weights = self.calculate_shallow_nn_weights(X, target_feature_index)
            else:
                raise ValueError(f"Unknown method: {method}")

            compromised_indices_set = set(compromised_indices)
            observed_data = [val for ind, val in enumerate(org_data) if ind not in compromised_indices_set]
            observed_X = np.delete(X, compromised_indices, axis=0)

            mean = np.mean(observed_data)
            a, b = min(observed_data), max(observed_data)
            t = len(compromised_indices)
            l = (b - a) / t
            V = [(a + (s - 1) * l + a + s * l) / 2 for s in range(1, t + 1)]

            R = []
            for v in V:
                contribution_weights = [1 - abs(i - v) / l if abs(i - v) <= l else 0 for i in observed_data]
                sum_weights = sum(contribution_weights)
                weighted_observed = 0.0
                for i in range(X.shape[1]):
                    if i != target_feature_index:
                        weighted_observed += feature_weights[i] * np.sum(observed_X[:, i] * contribution_weights)
                if sum_weights == 0:
                    R.append(mean)
                else:
                    R.append(weighted_observed / sum_weights)

            recovered_data = np.copy(org_data)
            for i, idx in enumerate(compromised_indices):
                recovered_data[idx] = R[i]
            return recovered_data

    def _gpu_basic_FID(self, org_data, compromised_indices):
        """Optimized GPU implementation of basic FID"""
        # conversion of inputs to tensors
        org_tensor = tf.convert_to_tensor(org_data, dtype=tf.float32)
        comp_indices = tf.convert_to_tensor(compromised_indices, dtype=tf.int32)

        # @tf.function

        # recovered_tensor = self.compute_fid(org_tensor, comp_indices)
        return self.compute_fid_tf(org_tensor, comp_indices).numpy()

    def compute_fid(self, org_tensor, comp_indices):
              # Creating mask for observed data
              mask = tf.ones_like(org_tensor, dtype=tf.bool)
              mask = tf.tensor_scatter_nd_update(
                  mask,
                  tf.expand_dims(comp_indices, 1),
                  tf.zeros_like(comp_indices, dtype=tf.bool))

              observed_data = tf.boolean_mask(org_tensor, mask)
              a, b = tf.reduce_min(observed_data), tf.reduce_max(observed_data)
              t = tf.size(comp_indices)
              l = (b - a) / tf.cast(t, tf.float32)

              s = tf.range(1, t+1, dtype=tf.float32)
              V = (a + (s-1)*l + a + s*l) / 2

              diff = tf.abs(tf.expand_dims(observed_data, 0) - tf.expand_dims(V, 1))
              weights = tf.where(diff <= l, 1 - diff/l, 0.0)

              sum_weights = tf.reduce_sum(weights, axis=1)
              weighted_sum = tf.reduce_sum(weights * tf.expand_dims(observed_data, 0), axis=1)
              R = tf.where(sum_weights > 0,
                          weighted_sum / sum_weights,
                          tf.reduce_mean(observed_data))

              return tf.tensor_scatter_nd_update(org_tensor,tf.expand_dims(comp_indices, 1),R)

    def _gpu_FID(self, org_data, compromised_indices, X=None, target_feature_index=None, method=None):
        if method is None:
            return self._gpu_basic_FID(org_data, compromised_indices)
        else:
            return self._gpu_correlation_FID(org_data, compromised_indices, X, target_feature_index, method)


    def _gpu_correlation_FID(self, org_data, compromised_indices, X, target_feature_index, method):
        org_tensor = tf.convert_to_tensor(org_data, dtype=tf.float32)
        X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
        comp_indices = tf.convert_to_tensor(compromised_indices, dtype=tf.int32)

        # 1. Calculate weights (GPU-optimized)
        if method == 'pearson':
            weights = self._gpu_pearson_correlation(X_tensor, target_feature_index)
        elif method == 'distance':
            weights = self._gpu_distance_correlation_weights(X_tensor, target_feature_index)
        elif method == 'shallow_nn':
            weights = self._gpu_shallow_nn(X_tensor, target_feature_index)

        # 2. Filter observed data
        observed_mask = tf.ones_like(org_tensor, dtype=tf.bool)
        observed_mask = tf.tensor_scatter_nd_update(
            observed_mask,
            tf.expand_dims(comp_indices, 1),
            tf.zeros_like(comp_indices, dtype=tf.bool)
        )
        observed_data = tf.boolean_mask(org_tensor, observed_mask)
        observed_X = tf.boolean_mask(X_tensor, observed_mask)

        # 3. Recovery intervals
        a, b = tf.reduce_min(observed_data), tf.reduce_max(observed_data)
        t = tf.size(comp_indices)
        l = (b - a) / tf.cast(t, tf.float32)
        s = tf.range(1, t+1, dtype=tf.float32)
        V = (a + (s-1)*l + a + s*l) / 2

        # 4. CRITICAL FIX: Apply weights like CPU version
        diff = tf.abs(tf.expand_dims(observed_data, 0) - tf.expand_dims(V, 1))  # Shape: [t, n_observed]
        contrib_weights = tf.where(diff <= l, 1 - diff/l, 0.0)  # Shape: [t, n_observed]

        # Vectorized equivalent of your CPU's weighted sum:
        # For each feature i: sum(observed_X[:,i] * contrib_weights[t,:]) * feature_weights[i]
        weighted_contributions = tf.einsum(
            'ji,tj->ti',  # observed_X.T [features x n_observed] @ contrib_weights [t x n_observed]
            observed_X * weights[None, :],  # Applying feature weights
            contrib_weights)

        # Sum across features (excluding target)
        mask = tf.one_hot(target_feature_index, X.shape[1], on_value=0.0, off_value=1.0)
        weighted_observed = tf.reduce_sum(weighted_contributions * mask, axis=1)  # Shape: [t]

        sum_weights = tf.reduce_sum(contrib_weights, axis=1)  # Shape: [t]
        R = tf.where(sum_weights > 0, weighted_observed / sum_weights, tf.reduce_mean(observed_data))


        return tf.tensor_scatter_nd_update(org_tensor,tf.expand_dims(comp_indices, 1),R).numpy()

In [ ]:
# evaluator = Evaluator()
# trainers = {}

# for feature in ['temp', 'RH']:
#   preprocessor = DataPreprocessor(df, feature, sigma=0.6, OP=20)
#   X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
#   print(f"For feature: {feature}")

#   feature_index = features.index(feature)
#   trainers[f"{feature}"] = ModelTrainer()

#   trainers[feature].train_rf(X_train, y_train, feature_index, feature)
#   trainers[feature].train_knn(X_train, y_train, feature_index, feature)
#   trainers[feature].train_autoencoder(X_train, X_test, preprocessor, feature_index, feature)
#   model_names = [f"rf_{feature}",f"knc_{feature}"]
#   # var_name3 = f"ae_{feature}"

#   #pending  (:p

#   combined_series = pd.concat([y_train, y_test])
#   # compromised_indices = np.where(combined_series == 0)[0]

#   com_indices = evaluator.evaluate(trainers[feature].models, X_test, feature_index, y_test)
#   # RF_compromised_indices = com_indices[f"rf_{feature}"]
#   # KNC_compromised_indices = com_indices[f"knc_{feature}"]
#   ytest_count_outliers = sum(1 for i, j in enumerate(y_test) if j == 0)

#   print(f"Actual  compromised data instances : {ytest_count_outliers}")
#   for key, value in com_indices.items():
#     print(f"For {key}, compromised data instances: {len(value)}")
#   print("\n")
#   # print(f"For  RF, compromised data instances: {len(RF_compromised_indices)}")
#   # print(f"For KNN, compromised data instances: {len(KNC_compromised_indices)}\n")
#   # print(f"For AE, compromised data instances: {len(AE_compromised_indices)}]\n")
#   print("**"*80,"\n\n")

In [ ]:
# # code block for collecting results of original vs recovered data correlation for both FID and modified FID.

import seaborn as sns
import matplotlib.pyplot as plt
import gc

def preprocess_pems(df):
    # Ensure datetime index (if not already)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.iloc[:, 0])  # Assuming first column is datetime
        df = df.iloc[:, 1:]  # Drop the original datetime column if duplicated

    df['minute'] = df.index.minute
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek  # Monday=0, Sunday=6
    df['is_weekend'] = df['day_of_week'] >= 5


    df = df.dropna().reset_index(drop=True)

    return df



# def analyze_datasets():

#     DATASETS = {
#         'forestfires': {
#             'file': '/content/drive/My Drive/MTP_DATA/forestfires.csv',
#             'features': ['temp', 'RH'],
#             'preprocess': lambda df: df  # No special preprocessing
#         }
#         # ,
#         # 'pems': {
#         #     'file': '/content/drive/My Drive/MTP_DATA/PEMS-BAY.csv',
#         #     'features': ['400001', '400017'],  # Sensor IDs from paper
#         #     'preprocess': preprocess_pems,  # Preprocessing function
#         #     'no_drop' : ['minute','hour','day_of_week','is_weekend']
#         # }
#         # ,
#         # 'unsw': {
#         #     'file': '/content/drive/My Drive/MTP_DATA/UNSW_NB15/UNSW_NB15_test.csv',
#         #     'features': ['dur', 'rate'],  # Features from paper
#         #     'preprocess': lambda df: df.drop(['state'], axis=1) #lambda df: df.select_dtypes(include=[np.number])  # Numerical only
#         # }
#     }


#     fid = FIDRecovery()


#     sigma_values = [0.2, 0.3, 0.4, 0.5]
#     outlier_percentages = [5, 10, 15, 20]
#     fixed_outlier_percentage = 15
#     fixed_sigma = 0.3

#     all_results = []

#     for dataset_name, config in DATASETS.items():
#         print(f"\n\t\t===  Processing {dataset_name} dataset  ===")

#         df = pd.read_csv(config['file'])
#         # print("Columns in dataset:", df.columns.tolist())
#         df = config['preprocess'](df)

#         # clearing any existing TensorFlow graphs to prevent memory buildup
#         tf.keras.backend.clear_session()
#         gc.collect()

#         dataset_results = {'sigma': [], 'outlier': []}
#         # if dataset_name == 'pems':
#         #     df = df[config['no_drop']+config['features']]

#         for feature in config['features']:
#             print(f"\nAnalyzing feature: {feature}")
#             if dataset_name == 'pems':
#                 features_to_keep = [feature] + config['no_drop']
#                 df_n = df[features_to_keep]
#             else:
#                 df_n = df

#             # Sigma variation analysis
#             for sigma in sigma_values:
#                 print(f"  Sigma: {sigma}", end=" | ")

#                 # Initialize preprocessor
#                 try:
#                     preprocessor = EnhancedDataPreprocessor(
#                         df= df_n, #if dataset_name=='pems' else df,
#                         dataset_name=dataset_name,
#                         sel_feature=feature,
#                         sigma=sigma,
#                         OP=fixed_outlier_percentage
#                     )
#                     # Prepare data with injected outliers
#                     X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
#                     feature_index = features.index(feature)
#                     compromised_indices = np.where(y_test == 0)[0]
#                     original_data = X_scaled[:, feature_index]

#                     # Test all recovery methods
#                     # methods = {
#                     #     'FID': fid.FID(original_data, compromised_indices),
#                     #     'FID + Pearson': fid.FID(original_data, compromised_indices,
#                     #                            X_scaled, feature_index, method='pearson'),
#                     #     # 'FID + Distance': fid.FID(original_data, compromised_indices,
#                     #     #                         X_scaled, feature_index, method='distance'),
#                     #     # 'FID + Distance': fid.FID_with_distance_correlation(original_data,
#                     #     #                                                   compromised_indices,
#                     #     #                                                   X_scaled, feature_index),
#                     #     'FID + NN': fid.FID(original_data, compromised_indices,
#                     #                             X_scaled, feature_index, method='shallow_nn'),
#                     # }

#                     # with explicit clearing intermediate tensors to tackle memory buildup
#                     methods = {}
#                     methods['FID'] = fid.FID(original_data, compromised_indices)

#                     # Clear intermediate tensors
#                     tf.keras.backend.clear_session()

#                     methods['FID + Pearson'] = fid.FID(
#                         original_data, compromised_indices,
#                         X_scaled, feature_index, method='pearson')

#                     tf.keras.backend.clear_session()

#                     methods['FID + NN'] = fid.FID(
#                         original_data, compromised_indices,
#                         X_scaled, feature_index, method='shallow_nn')

#                     # Store results
#                     for method_name, recovered in methods.items():
#                         corr, _ = pearsonr(original_data, recovered)
#                         dataset_results['sigma'].append({
#                             'dataset': dataset_name,
#                             'feature': feature,
#                             'method': method_name,
#                             'sigma': sigma,
#                             'correlation': corr
#                         })

#                     # Explicit cleanup
#                     del X_scaled, X_train, X_test, y_train, y_test, original_data
#                     tf.keras.backend.clear_session()

#                     print("Done")

#                 except Exception as e:
#                     print(f"Error processing sigma {sigma}: {str(e)}")
#                     continue

#             # Outlier percentage variation analysis
#             for OP in outlier_percentages:
#                 print(f"  Outlier %: {OP}", end=" | ")

#                 try:
#                     preprocessor = EnhancedDataPreprocessor(
#                         df=df_n, #if dataset_name=='pems' else df,
#                         dataset_name=dataset_name,
#                         sel_feature=feature,
#                         sigma=fixed_sigma,
#                         OP=OP
#                     )

#                     X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
#                     feature_index = features.index(feature)
#                     compromised_indices = np.where(y_test == 0)[0]
#                     original_data = X_scaled[:, feature_index]

#                     # methods = {
#                     #     'FID': fid.FID(original_data, compromised_indices),
#                     #     'FID + Pearson': fid.FID(original_data, compromised_indices,
#                     #                            X_scaled, feature_index, method='pearson'),
#                     #     # 'FID + Distance': fid.FID(original_data, compromised_indices,
#                     #     #                         X_scaled, feature_index, method='distance'),
#                     #     # 'FID + Distance': fid.FID_with_distance_correlation(original_data,
#                     #     #                                                   compromised_indices,
#                     #     #                                                   X_scaled, feature_index),
#                     #     'FID + NN': fid.FID(original_data, compromised_indices,
#                     #                             X_scaled, feature_index, method='shallow_nn'),
#                     # }

#                     #with explicit clearing intermediate tensors to tackle memory buildup
#                     methods = {}
#                     methods['FID'] = fid.FID(original_data, compromised_indices)

#                     # Clear intermediate tensors
#                     tf.keras.backend.clear_session()

#                     # Process Pearson version
#                     methods['FID + Pearson'] = fid.FID(
#                         original_data, compromised_indices,
#                         X_scaled, feature_index, method='pearson')

#                     tf.keras.backend.clear_session()

#                     # Process NN version
#                     methods['FID + NN'] = fid.FID(
#                         original_data, compromised_indices,
#                         X_scaled, feature_index, method='shallow_nn')

#                     for method_name, recovered in methods.items():
#                         corr, _ = pearsonr(original_data, recovered)
#                         dataset_results['outlier'].append({
#                             'dataset': dataset_name,
#                             'feature': feature,
#                             'method': method_name,
#                             'outlier_percentage': OP,
#                             'correlation': corr
#                         })
#                     # Explicit cleanup
#                     del X_scaled, X_train, X_test, y_train, y_test, original_data
#                     tf.keras.backend.clear_session()

#                     print("Done")

#                 except Exception as e:
#                     print(f"Error processing sigma {sigma}: {str(e)}")
#                     continue

#         # Convert to DataFrames
#         sigma_df = pd.DataFrame(dataset_results['sigma'])
#         outlier_df = pd.DataFrame(dataset_results['outlier'])
#         all_results.append((sigma_df, outlier_df))

#     return all_results

# # Run analysis and visualize results
# def visualize_results(all_results):
#     for sigma_df, outlier_df in all_results:
#         dataset_name = sigma_df['dataset'].iloc[0]

#         # Sigma vs Correlation plot
#         plt.figure(figsize=(12, 6))
#         sns.lineplot(data=sigma_df, x='sigma', y='correlation', hue='method', style='feature')
#         plt.title(f'{dataset_name}: Sigma vs Correlation')
#         plt.grid(True)
#         plt.show()

#         # Outlier Percentage vs Correlation plot
#         plt.figure(figsize=(12, 6))
#         sns.lineplot(data=outlier_df, x='outlier_percentage', y='correlation', hue='method', style='feature')
#         plt.title(f'{dataset_name}: Outlier Percentage vs Correlation')
#         plt.grid(True)
#         plt.show()


# results = analyze_datasets()
# visualize_results(results)

In [ ]:
# # code block for collecting results of ML models accuracy for originally noised, FID recovered and FID+NN recovered data.

# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import accuracy_score


# sigma_values = [0.2, 0.3, 0.4, 0.5]
# outlier_percentages = [5, 10, 15, 20]
# fixed_outlier_percentage = 15
# fixed_sigma = 0.3

# def analyze_with_models():
#     DATASETS = {
#         'forestfires': {
#             'file': '/content/drive/My Drive/MTP_DATA/forestfires.csv',
#             'features': ['temp', 'RH'],
#             'preprocess': lambda df: df
#         }
#     }

#     fid = FIDRecovery()
#     all_results = []

#     for dataset_name, config in DATASETS.items():
#         print(f"\nProcessing {dataset_name} dataset")
#         df = pd.read_csv(config['file'])
#         df = config['preprocess'](df)
#         tf.keras.backend.clear_session()
#         gc.collect()

#         dataset_results = {
#             'sigma': [],
#             'outlier_percentage': [],
#             'feature': [],
#             'data_type': [],  # 'original', 'fid', 'fid_nn'
#             'rf_accuracy': [],
#             'knn_accuracy': [],
#             'ae_accuracy': []
#         }

#         for feature in config['features']:
#             # Sigma variation analysis
#             for sigma in sigma_values:
#                     print(f"  Sigma: {sigma}", end=" | ")
#                 # try:
#                     preprocessor = EnhancedDataPreprocessor(
#                         df=df, dataset_name=dataset_name,
#                         sel_feature=feature, sigma=sigma,
#                         OP=fixed_outlier_percentage
#                     )
#                     X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
#                     feature_index = features.index(feature)
#                     compromised_indices = np.where(y_test == 0)[0]

#                     # Train models
#                     trainer = ModelTrainer()
#                     # Smart training/loading
#                     trainer.train_or_load(X_train, y_train, dataset_name, feature, 'rf', sigma, fixed_outlier_percentage)
#                     trainer.train_or_load(X_train, y_train, dataset_name, feature, 'knn', sigma, fixed_outlier_percentage)
#                     trainer.train_or_load(X_train, y_train, dataset_name, feature, 'ae', sigma, fixed_outlier_percentage)

#                     # Evaluate on original data (with outliers)
#                     orig_rf_acc = accuracy_score(y_test, trainer.models[f"rf_{feature}"].predict(X_test))
#                     orig_knn_acc = accuracy_score(y_test, trainer.models[f"knn_{feature}"].predict(X_test))
#                     orig_ae_acc = accuracy_score(y_test, trainer.models[f"ae_{feature}"].predict_label(X_test))

#                     # FID recovery
#                     recovered_fid = fid.FID(X_scaled[:, feature_index], compromised_indices)
#                     X_test_fid = X_test.copy()
#                     X_test_fid[compromised_indices, feature_index] = recovered_fid[compromised_indices]

#                     # FID+NN recovery
#                     tf.keras.backend.clear_session()
#                     recovered_fid_nn = fid.FID(X_scaled[:, feature_index], compromised_indices,
#                                              X_scaled, feature_index, method='shallow_nn')
#                     X_test_fid_nn = X_test.copy()
#                     X_test_fid_nn[compromised_indices, feature_index] = recovered_fid_nn[compromised_indices]

#                     # Evaluate recovered data
#                     def evaluate_recovered(X_test_rec):
#                         return {
#                             'rf': accuracy_score(y_test, trainer.models[f"rf_{feature}"].predict(X_test_rec)),
#                             'knn': accuracy_score(y_test, trainer.models[f"knn_{feature}"].predict(X_test_rec)),
#                             'ae': accuracy_score(y_test, trainer.models[f"ae_{feature}"].predict_label(X_test_rec))
#                         }

#                     fid_results = evaluate_recovered(X_test_fid)
#                     fid_nn_results = evaluate_recovered(X_test_fid_nn)

#                     # Store results
#                     for data_type, results in zip(
#                         ['original', 'fid', 'fid_nn'],
#                         [
#                             {'rf': orig_rf_acc, 'knn': orig_knn_acc, 'ae': orig_ae_acc},
#                             fid_results,
#                             fid_nn_results
#                         ]
#                     ):
#                         dataset_results['sigma'].append(sigma)
#                         dataset_results['outlier_percentage'].append(fixed_outlier_percentage)
#                         dataset_results['feature'].append(feature)
#                         dataset_results['data_type'].append(data_type)
#                         dataset_results['rf_accuracy'].append(results['rf'])
#                         dataset_results['knn_accuracy'].append(results['knn'])
#                         dataset_results['ae_accuracy'].append(results['ae'])

#                      # Explicit cleanup
#                     del X_scaled, X_train, X_test, y_train, y_test
#                     tf.keras.backend.clear_session()

#                     print("\nDone\n")
#                 # except Exception as e:
#                 #     print(f"\nError: {str(e)}\n")
#                 #     continue

#             # Outlier percentage variation analysis
#             for OP in outlier_percentages:
#                 print(f"  Outlier %: {OP}", end=" | ")
#                 try:
#                     preprocessor = EnhancedDataPreprocessor(
#                         df=df, dataset_name=dataset_name,
#                         sel_feature=feature, sigma=fixed_sigma,
#                         OP=OP
#                     )
#                     X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
#                     feature_index = features.index(feature)
#                     compromised_indices = np.where(y_test == 0)[0]

#                     # Train models
#                     trainer = ModelTrainer()
#                     # Smart training/loading
#                     trainer.train_or_load(X_train, y_train, dataset_name, feature, 'rf', fixed_sigma, OP)
#                     trainer.train_or_load(X_train, y_train, dataset_name, feature, 'knn', fixed_sigma, OP)
#                     trainer.train_or_load(X_train, y_train, dataset_name, feature, 'ae', fixed_sigma, OP)

#                     # Original evaluation
#                     orig_rf_acc = accuracy_score(y_test, trainer.models[f"rf_{feature}"].predict(X_test))
#                     orig_knn_acc = accuracy_score(y_test, trainer.models[f"knn_{feature}"].predict(X_test))
#                     orig_ae_acc = accuracy_score(y_test, trainer.models[f"ae_{feature}"].predict_label(X_test))

#                     # Recover data
#                     recovered_fid = fid.FID(X_scaled[:, feature_index], compromised_indices)
#                     X_test_fid = X_test.copy()
#                     X_test_fid[compromised_indices, feature_index] = recovered_fid[compromised_indices]

#                     tf.keras.backend.clear_session()
#                     recovered_fid_nn = fid.FID(X_scaled[:, feature_index], compromised_indices,
#                                              X_scaled, feature_index, method='shallow_nn')
#                     X_test_fid_nn = X_test.copy()
#                     X_test_fid_nn[compromised_indices, feature_index] = recovered_fid_nn[compromised_indices]

#                     # Evaluate recovered
#                     fid_results = evaluate_recovered(X_test_fid)
#                     fid_nn_results = evaluate_recovered(X_test_fid_nn)

#                     # Store results
#                     for data_type, results in zip(
#                         ['original', 'fid', 'fid_nn'],
#                         [
#                             {'rf': orig_rf_acc, 'knn': orig_knn_acc, 'ae': orig_ae_acc},
#                             fid_results,
#                             fid_nn_results
#                         ]
#                     ):
#                         dataset_results['sigma'].append(fixed_sigma)
#                         dataset_results['outlier_percentage'].append(OP)
#                         dataset_results['feature'].append(feature)
#                         dataset_results['data_type'].append(data_type)
#                         dataset_results['rf_accuracy'].append(results['rf'])
#                         dataset_results['knn_accuracy'].append(results['knn'])
#                         dataset_results['ae_accuracy'].append(results['ae'])

#                      # Explicit cleanup
#                     del X_scaled, X_train, X_test, y_train, y_test
#                     tf.keras.backend.clear_session()

#                     print("\nDone\n")
#                 except Exception as e:
#                     print(f"\nError: {str(e)}\n")
#                     continue

#         all_results.append(pd.DataFrame(dataset_results))

#     return all_results

# def plot_model_comparisons(results):
#     for df in results:
#         dataset_name = df['feature'].iloc[0].split('_')[0] if '_' in df['feature'].iloc[0] else df['feature'].iloc[0]

#         # Plot for sigma variations
#         plt.figure(figsize=(15, 5))
#         plt.subplot(1, 2, 1)
#         sns.lineplot(
#             data=df[df['outlier_percentage'] == fixed_outlier_percentage],
#             x='sigma', y='rf_accuracy', hue='data_type',
#             style='feature', markers=True, dashes=False
#         )
#         plt.title('RF Accuracy: Sigma Variations')
#         plt.grid(True)

#         plt.subplot(1, 2, 2)
#         sns.lineplot(
#             data=df[df['sigma'] == fixed_sigma],
#             x='outlier_percentage', y='rf_accuracy', hue='data_type',
#             style='feature', markers=True, dashes=False
#         )
#         plt.title('RF Accuracy: Outlier Percentage Variations')
#         plt.grid(True)
#         plt.tight_layout()
#         plt.show()

#         # Repeat similar plots for KNN and AE
#         # ... (similar code blocks for other models) ...

# # Run analysis
# model_results = analyze_with_models()
# plot_model_comparisons(model_results)

# # Print performance tables
# for df in model_results:
#     print("\nPerformance Summary:")
#     print(df.groupby(['feature', 'data_type'])[['rf_accuracy', 'knn_accuracy', 'ae_accuracy']].mean())

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, ks_2samp
from scipy.spatial.distance import jensenshannon
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

class RecoveryEvaluator:
    def __init__(self, original_data, features):
        self.original_data = original_data
        self.features = features
        self.results = {}

    def evaluate_recovery(self, recovered_data, compromised_indices, method_name, X_train=None, y_train=None):
        """Main evaluation method to assess recovery quality"""
        self.results[method_name] = {}

        # 1. Direct Reconstruction Metrics
        self.results[method_name]['reconstruction'] = self._calculate_reconstruction_metrics(
            self.original_data, recovered_data, compromised_indices)

        # 2. Statistical Property Preservation
        self.results[method_name]['statistics'] = self._compare_distributions(
            self.original_data, recovered_data)

        # 3. Downstream Task Performance (if labels available)
        # if X_train is not None and y_train is not None:
        #     self.results[method_name]['downstream'] = self._evaluate_downstream_performance(
        #         X_train, y_train, recovered_data)

        # 4. Visual Assessment
        # self._plot_recovery_comparison(self.original_data, recovered_data,
        #                              compromised_indices, method_name)

        return self.results[method_name]


    def safe_ks_test(x, y):
        """Handle KS test warnings and edge cases"""
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            try:
                result = ks_2samp(x, y)
                return result.statistic
            except:
                return np.nan

    def _calculate_reconstruction_metrics(self, original, recovered, compromised_idx):
        """Calculate metrics for compromised points only"""
        comp_original = original[compromised_idx]
        comp_recovered = recovered[compromised_idx]
        metrics = {}
        try:
          if tf.config.list_physical_devices('GPU'):
            fid = FIDRecovery()
            metrics['Pearson_r'] = fid._gpu_pearson_correlation(comp_original, comp_recovered)
          else:
            metrics['Pearson_r'] = pearsonr(comp_original, comp_recovered)[0]
        except:
            metrics['Pearson_r'] = np.nan

        # try:
        #     metrics['Jensen-Shannon'] = jensenshannon(comp_original, comp_recovered)
        # except:
        #     metrics['Jensen-Shannon'] = np.nan

        # metrics['ks_statistic'] =self. safe_ks_test(comp_original, comp_recovered)

        metrics['MAE'] = mean_absolute_error(comp_original, comp_recovered),
        metrics['RMSE'] = np.sqrt(mean_squared_error(comp_original, comp_recovered)),
        # metrics['Pearson_r'] = pearsonr(comp_original, comp_recovered)[0],
        # metrics['Jensen-Shannon'] = jensenshannon(comp_original, comp_recovered),
        metrics['Recovery_Rate'] = np.mean(np.abs(comp_original - comp_recovered)) /(np.abs(comp_original).mean() + 1e-8)
        return metrics


    def _compare_distributions(self, original, recovered):
        """Compare overall statistical properties"""
        return {
            'KS_Statistic': ks_2samp(original, recovered).statistic,
            'Mean_Difference': np.mean(original) - np.mean(recovered),
            'Std_Difference': np.std(original) - np.std(recovered),
            'Median_Difference': np.median(original) - np.median(recovered)
        }

    def _evaluate_downstream_performance(self, X_train, y_train, recovered_data):
        """Train model on original data, test on recovered"""
        # Split original data for training
        X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.3)

        # Train model
        model = RandomForestClassifier(n_estimators=100)
        model.fit(X_tr, y_tr)

        # Test on original test set (baseline)
        baseline_score = model.score(X_te, y_te)

        # Test on recovered data
        recovered_score = model.score(recovered_data, y_te)

        return {
            'baseline_accuracy': baseline_score,
            'recovered_accuracy': recovered_score,
            'performance_gap': baseline_score - recovered_score
        }

    def _plot_recovery_comparison(self, original, recovered, compromised_idx, method_name):
        """Visual comparison of original vs recovered data"""
        plt.figure(figsize=(15, 5))

        # Time series plot
        plt.subplot(1, 2, 1)
        plt.plot(original, 'b-', alpha=0.3, label='Original')
        plt.plot(recovered, 'r--', alpha=0.5, label='Recovered')
        plt.scatter(compromised_idx, original[compromised_idx],
                   c='black', marker='x', s=50, label='Compromised')
        plt.title(f'{method_name} Recovery Comparison')
        plt.legend()

        # Distribution plot
        plt.subplot(1, 2, 2)
        sns.kdeplot(original, label='Original', fill=True)
        sns.kdeplot(recovered, label='Recovered', fill=True)
        plt.title('Distribution Comparison')
        plt.legend()

        plt.tight_layout()
        plt.show()

    def generate_report(self):
        """Generate a comprehensive comparison report"""
        report = {}
        for method, metrics in self.results.items():
            report[method] = {
                'Reconstruction_Quality': metrics['reconstruction'],
                'Statistical_Similarity': metrics['statistics']
            }
            # if 'downstream' in metrics:
            #     report[method]['Downstream_Performance'] = metrics['downstream']

        return pd.DataFrame(report).T








sigma_values = [0.2, 0.3, 0.4, 0.5]
outlier_percentages = [5, 10, 15, 20]
fixed_outlier_percentage = 15
fixed_sigma = 0.3

def analyze_with_recovery_metrics():
    DATASETS = {
        # 'forestfires': {
        #     'file': '/content/drive/My Drive/MTP_DATA/forestfires.csv',
        #     'features': ['temp', 'RH'],
        #     'preprocess': lambda df: df
        # }
        #         ,
        'pems': {
            'file': '/content/drive/My Drive/MTP_DATA/PEMS-BAY.csv',
            'features': ['400001', '400017'],  # Sensor IDs from paper
            'preprocess': preprocess_pems,  # Preprocessing function
            'no_drop' : ['minute','hour','day_of_week','is_weekend']
        }
    }

    fid = FIDRecovery()
    all_results = []

    for dataset_name, config in DATASETS.items():
        print(f"\nProcessing {dataset_name} dataset")
        df = pd.read_csv(config['file'])
        df = config['preprocess'](df)

        dataset_results = {
            'dataset': dataset_name, #* len(sigma_values) * len(outlier_percentages),
            'sigma': [],
            'outlier_percentage': [],
            'feature': [],
            'method': [],  # 'fid' or 'fid_nn'
            'mae': [],
            'rmse': [],
            'pearson_r': [],
            'js_divergence': [],
            'ks_statistic': [],
            'mean_diff': [],
            'std_diff': []
        }

        for feature in config['features']:
            # Sigma variation analysis
            for sigma in sigma_values:
                print(f"  Sigma: {sigma}", end=" | ")
                try:
                    preprocessor = EnhancedDataPreprocessor(
                        df=df, dataset_name=dataset_name,
                        sel_feature=feature, sigma=sigma,
                        OP=fixed_outlier_percentage
                    )
                    X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
                    feature_index = features.index(feature)
                    compromised_indices = np.where(y_test == 0)[0]
                    original_values = X_scaled[:, feature_index]

                    # Initialize evaluator with original data
                    evaluator = RecoveryEvaluator(original_data=original_values, features=features)

                    # FID recovery
                    recovered_fid = fid.FID(original_values, compromised_indices)
                    metrics_fid = evaluator._calculate_reconstruction_metrics(original_values, recovered_fid, compromised_indices)
                    stats_fid = evaluator._compare_distributions(original_values, recovered_fid)

                    # FID+NN recovery
                    # tf.keras.backend.clear_session()
                    recovered_fid_nn = fid.FID(original_values, compromised_indices,
                                             X_scaled, feature_index, method='shallow_nn')
                    metrics_nn = evaluator._calculate_reconstruction_metrics(original_values, recovered_fid_nn, compromised_indices)
                    stats_nn = evaluator._compare_distributions(original_values, recovered_fid_nn)

                    # Store FID results
                    dataset_results['sigma'].append(sigma)
                    dataset_results['outlier_percentage'].append(fixed_outlier_percentage)
                    dataset_results['feature'].append(feature)
                    dataset_results['method'].append('fid')
                    dataset_results['mae'].append(metrics_fid['MAE'])
                    dataset_results['rmse'].append(metrics_fid['RMSE'])
                    dataset_results['pearson_r'].append(metrics_fid['Pearson_r'])
                    # dataset_results['js_divergence'].append(metrics_fid['Jensen-Shannon'])
                    dataset_results['ks_statistic'].append(stats_fid['KS_Statistic'])
                    dataset_results['mean_diff'].append(stats_fid['Mean_Difference'])
                    dataset_results['std_diff'].append(stats_fid['Std_Difference'])

                    # Store FID+NN results
                    dataset_results['sigma'].append(sigma)
                    dataset_results['outlier_percentage'].append(fixed_outlier_percentage)
                    dataset_results['feature'].append(feature)
                    dataset_results['method'].append('fid_nn')
                    dataset_results['mae'].append(metrics_nn['MAE'])
                    dataset_results['rmse'].append(metrics_nn['RMSE'])
                    dataset_results['pearson_r'].append(metrics_nn['Pearson_r'])
                    # dataset_results['js_divergence'].append(metrics_nn['Jensen-Shannon'])
                    dataset_results['ks_statistic'].append(stats_nn['KS_Statistic'])
                    dataset_results['mean_diff'].append(stats_nn['Mean_Difference'])
                    dataset_results['std_diff'].append(stats_nn['Std_Difference'])

                    # Generate visual comparison
                    # evaluator._plot_recovery_comparison(
                    #     original_values, recovered_fid, compromised_indices,
                    #     f"FID (σ={sigma}, {feature})")
                    # evaluator._plot_recovery_comparison(
                    #     original_values, recovered_fid_nn, compromised_indices,
                    #     f"FID+NN (σ={sigma}, {feature})")

                    # Cleanup
                    del X_scaled, X_train, X_test, y_train, y_test
                    # tf.keras.backend.clear_session()
                    # gc.collect()
                    clear_gpu_memory()

                    print("Done")
                except Exception as e:
                    print(f"Error: {str(e)}")
                    continue

            # Outlier percentage variation analysis
            for OP in outlier_percentages:
                print(f"  Outlier %: {OP}", end=" | ")
                try:
                    preprocessor = EnhancedDataPreprocessor(
                        df=df, dataset_name=dataset_name,
                        sel_feature=feature, sigma=fixed_sigma,
                        OP=OP
                    )
                    X_scaled, X_train, X_test, y_train, y_test, features = preprocessor.prepare_data()
                    feature_index = features.index(feature)
                    compromised_indices = np.where(y_test == 0)[0]
                    original_values = X_scaled[:, feature_index]

                    # Initialize evaluator
                    evaluator = RecoveryEvaluator(original_data=original_values, features=features)

                    # FID recovery
                    recovered_fid = fid.FID(original_values, compromised_indices)
                    metrics_fid = evaluator._calculate_reconstruction_metrics(
                        original_values, recovered_fid, compromised_indices)
                    stats_fid = evaluator._compare_distributions(original_values, recovered_fid)

                    # FID+NN recovery
                    # tf.keras.backend.clear_session()
                    recovered_fid_nn = fid.FID(original_values, compromised_indices,
                                             X_scaled, feature_index, method='shallow_nn')
                    metrics_nn = evaluator._calculate_reconstruction_metrics(
                        original_values, recovered_fid_nn, compromised_indices)
                    stats_nn = evaluator._compare_distributions(original_values, recovered_fid_nn)

                    # Store FID results
                    dataset_results['sigma'].append(fixed_sigma)
                    dataset_results['outlier_percentage'].append(OP)
                    dataset_results['feature'].append(feature)
                    dataset_results['method'].append('fid')
                    dataset_results['mae'].append(metrics_fid['MAE'])
                    dataset_results['rmse'].append(metrics_fid['RMSE'])
                    dataset_results['pearson_r'].append(metrics_fid['Pearson_r'])
                    # dataset_results['js_divergence'].append(metrics_fid['Jensen-Shannon'])
                    dataset_results['ks_statistic'].append(stats_fid['KS_Statistic'])
                    dataset_results['mean_diff'].append(stats_fid['Mean_Difference'])
                    dataset_results['std_diff'].append(stats_fid['Std_Difference'])

                    # Store FID+NN results
                    dataset_results['sigma'].append(fixed_sigma)
                    dataset_results['outlier_percentage'].append(OP)
                    dataset_results['feature'].append(feature)
                    dataset_results['method'].append('fid_nn')
                    dataset_results['mae'].append(metrics_nn['MAE'])
                    dataset_results['rmse'].append(metrics_nn['RMSE'])
                    dataset_results['pearson_r'].append(metrics_nn['Pearson_r'])
                    # dataset_results['js_divergence'].append(metrics_nn['Jensen-Shannon'])
                    dataset_results['ks_statistic'].append(stats_nn['KS_Statistic'])
                    dataset_results['mean_diff'].append(stats_nn['Mean_Difference'])
                    dataset_results['std_diff'].append(stats_nn['Std_Difference'])

                    # Cleanup
                    del X_scaled, X_train, X_test, y_train, y_test
                    # tf.keras.backend.clear_session()
                    # gc.collect()
                    clear_gpu_memory()

                    print("Done")
                except Exception as e:
                    print(f"Error: {str(e)}")
                    continue

        all_results.append(pd.DataFrame(dataset_results))

    return all_results

In [ ]:
def plot_metric_trends(results_df, metric='mae'):
    """Plot metric trends across all datasets and features"""
    g = sns.FacetGrid(results_df,
                    col='dataset',
                    row='feature',
                    hue='method',
                    height=4,
                    aspect=1.2,
                    sharey=False)

    g.map(sns.lineplot, 'sigma', metric, marker='o')
    g.add_legend()
    g.set_titles(col_template="{col_name}", row_template="{row_name}")
    g.set_axis_labels("Sigma", metric.upper())
    plt.subplots_adjust(top=0.9)
    g.fig.suptitle(f'{metric.upper()} Trends Across Datasets')
    plt.show()

def plot_op_trends(results_df, metric='rmse'):
    """Plot outlier percentage trends across datasets"""
    g = sns.FacetGrid(results_df[results_df['sigma'] == fixed_sigma],
                    col='dataset',
                    row='feature',
                    hue='method',
                    height=4,
                    aspect=1.2,
                    sharey=False)

    g.map(sns.lineplot, 'outlier_percentage', metric, marker='o')
    g.add_legend()
    g.set_titles(col_template="{col_name}", row_template="{row_name}")
    g.set_axis_labels("Outlier Percentage", metric.upper())
    plt.subplots_adjust(top=0.9)
    g.fig.suptitle(f'{metric.upper()} vs Outlier Percentage')
    plt.show()

def plot_method_comparison(results_df):
    """Compare methods across datasets using bar plots"""
    metrics = ['mae', 'rmse', 'pearson_r']

    for metric in metrics:
        plt.figure(figsize=(12, 6))
        sns.barplot(data=results_df,
                   x='dataset',
                   y=metric,
                   hue='method',
                   ci='sd')
        plt.title(f'{metric.upper()} Comparison Across Datasets')
        plt.ylabel(metric.upper())
        plt.xlabel('Dataset')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.show()


def plot_statistical_comparisons(results_df):
    """Compare statistical properties across methods"""
    stats = ['mean_diff', 'std_diff', 'ks_statistic']

    for stat in stats:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=results_df,
                  x='dataset',
                  y=stat,
                  hue='method')
        plt.title(f'{stat.replace("_", " ").title()} Comparison')
        plt.ylabel(stat.replace("_", " ").title())
        plt.xlabel('Dataset')
        plt.axhline(0, color='gray', linestyle='--')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.show()


def generate_all_plots(results_df):
    """Generate complete set of analysis plots"""
    # 1. Metric trends across sigma values
    for metric in ['mae', 'rmse', 'pearson_r']:
        plot_metric_trends(results_df, metric=metric)

    # 2. Outlier percentage trends
    for metric in ['mae', 'rmse', 'js_divergence']:
        plot_op_trends(results_df, metric=metric)

    # 3. Method comparison across datasets
    plot_method_comparison(results_df)

    # 4. Statistical property comparisons
    plot_statistical_comparisons(results_df)

In [ ]:
# Run analysis
final_results = analyze_with_recovery_metrics()

# Generate all plots
# generate_all_plots(pd.concat(final_results, ignore_index=True))

# Optional: Save results for later analysis
# final_results.to_csv('recovery_metrics_all_datasets.csv', index=False)


Processing pems dataset
  Sigma: 0.2 | Done
  Sigma: 0.3 | Done
  Sigma: 0.4 | Done
  Sigma: 0.5 | Done
  Outlier %: 5 | 

In [ ]:
import pickle

# Save
# with open('/content/drive/My Drive/MTP_DATA/results_of_unsw1.pkl', 'wb') as f:
#     pickle.dump(results, f)

# Load
# with open('/content/drive/My Drive/MTP_DATA/results_of_unsw.pkl', 'rb') as f:
#     loaded_data = pickle.load(f)
#     print(loaded_data)

In [ ]:
# # Print the actual correlation values for comparison
# for sigma_df, outlier_df in results:
#     print("\nSigma variations:")
#     print(sigma_df.groupby(['feature', 'method'])['correlation'].mean())

#     print("\nOutlier variations:")
#     print(outlier_df.groupby(['feature', 'method'])['correlation'].mean())

In [ ]:
# def visualize_results2(all_results):
#     for sigma_df, outlier_df in all_results:
#         dataset_name = sigma_df['dataset'].iloc[0]

#         # Get unique features and methods
#         features = sigma_df['feature'].unique()
#         methods = sigma_df['method'].unique()

#         # Create a color palette
#         palette = sns.color_palette("husl", len(methods))

#         # Plot for each feature separately
#         for feature in features:
#             # Filter data for current feature
#             feature_sigma_df = sigma_df[sigma_df['feature'] == feature]
#             feature_outlier_df = outlier_df[outlier_df['feature'] == feature]

#             # Create figure with subplots
#             fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
#             fig.suptitle(f'{dataset_name} - Feature: {feature}', fontsize=14)

#             # Add jitter to x-axis values
#             jitter_amount = 0.02  # Adjust as needed

#             # Plot 1: Sigma vs Correlation
#             for i, method in enumerate(methods):
#                 method_df = feature_sigma_df[feature_sigma_df['method'] == method]
#                 # Add small jitter to x values
#                 jittered_sigma = method_df['sigma'] + np.random.uniform(-jitter_amount, jitter_amount, len(method_df))
#                 sns.lineplot(
#                     x=jittered_sigma,
#                     y=method_df['correlation'],
#                     color=palette[i],
#                     label=method,
#                     ax=ax1,
#                     linewidth=2.5,
#                     alpha=0.8
#                 )
#                 # Add scatter points
#                 sns.scatterplot(
#                     x=jittered_sigma,
#                     y=method_df['correlation'],
#                     color=palette[i],
#                     ax=ax1,
#                     s=80,
#                     edgecolor='w'
#                 )

#             ax1.set_title('Sigma vs Correlation')
#             ax1.set_xlabel('Sigma')
#             ax1.set_ylabel('Correlation')
#             ax1.grid(True, alpha=0.3)

#             # Plot 2: Outlier Percentage vs Correlation
#             for i, method in enumerate(methods):
#                 method_df = feature_outlier_df[feature_outlier_df['method'] == method]
#                 # Add small jitter to x values
#                 jittered_op = method_df['outlier_percentage'] + np.random.uniform(-jitter_amount, jitter_amount, len(method_df))
#                 sns.lineplot(
#                     x=jittered_op,
#                     y=method_df['correlation'],
#                     color=palette[i],
#                     label=method,
#                     ax=ax2,
#                     linewidth=2.5,
#                     alpha=0.8
#                 )
#                 # Add scatter points
#                 sns.scatterplot(
#                     x=jittered_op,
#                     y=method_df['correlation'],
#                     color=palette[i],
#                     ax=ax2,
#                     s=80,
#                     edgecolor='w'
#                 )

#             ax2.set_title('Outlier Percentage vs Correlation')
#             ax2.set_xlabel('Outlier Percentage')
#             ax2.set_ylabel('Correlation')
#             ax2.grid(True, alpha=0.3)

#             # Common adjustments
#             for ax in [ax1, ax2]:
#                 ax.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
#                 ax.set_ylim(0, 1.1)  # Correlation ranges from 0 to 1

#             plt.tight_layout()
#             plt.show()
# visualize_results2(results)

In [ ]:
def visualize_results3(all_results):
    for sigma_df, outlier_df in all_results:
        dataset_name = sigma_df['dataset'].iloc[0]
        features = sigma_df['feature'].unique()

        # Configuration
        jitter_strength = 0.0019  # Optimal from your previous version
        marker_size = 6
        line_width = 1.5
        alpha = 0.9

        for feature in features:
            # Prepare data for current feature
            feature_sigma_df = sigma_df[sigma_df['feature'] == feature].copy()
            feature_outlier_df = outlier_df[outlier_df['feature'] == feature].copy()

            # Apply your smart jitter logic only to FID variants
            for df in [feature_sigma_df, feature_outlier_df]:
                df['correlation_jittered'] = df['correlation'] + df['method'].apply(
                    lambda m: np.random.uniform(-jitter_strength, jitter_strength)
                    if 'FID' in m and m != 'FID' else 0
                )

            # Create figure
            plt.figure(figsize=(16, 6))
            plt.suptitle(f'{dataset_name} - Feature: {feature}', y=1.02, fontsize=14)

            # ===== Sigma vs Correlation =====
            plt.subplot(1, 2, 1)
            sns.lineplot(
                data=feature_sigma_df,
                x='sigma',
                y='correlation_jittered',
                hue='method',
                style='method',
                markers=True,
                dashes=False,
                markersize=marker_size,
                linewidth=line_width,
                alpha=alpha,
                markeredgecolor='w'
            )
            plt.title('Sigma vs Correlation', pad=15)
            plt.xlabel('Sigma', labelpad=10)
            plt.ylabel('Correlation', labelpad=10)
            plt.grid(True, alpha=0.2)
            plt.ylim(0, 1.1)

            # ===== Outlier Percentage vs Correlation =====
            plt.subplot(1, 2, 2)
            sns.lineplot(
                data=feature_outlier_df,
                x='outlier_percentage',
                y='correlation_jittered',
                hue='method',
                style='method',
                markers=True,
                dashes=False,
                markersize=marker_size,
                linewidth=line_width,
                alpha=alpha,
                markeredgecolor='w'
            )
            plt.title('Outlier Percentage vs Correlation', pad=15)
            plt.xlabel('Outlier Percentage', labelpad=10)
            plt.ylabel('Correlation', labelpad=10)
            plt.grid(True, alpha=0.2)
            plt.ylim(0, 1.1)

            # Adjust legend
            handles, labels = plt.gca().get_legend_handles_labels()
            plt.figlegend(
                handles, labels,
                loc='upper center',
                ncol=min(4, len(labels)),  # Max 4 columns
                bbox_to_anchor=(0.5, 1.02),
                frameon=False
            )

            plt.tight_layout()
            plt.show()

visualize_results3(results)